## Root Cause Analysis - Exp2

* Goals
  * Root Cause Analysis on individual RAG results --> 2 RAGs results comparison
  * Provide actionable suggestions
* About Exp2
  * Build Langgraph Agent System for Root Cause Analysis

In [8]:
%load_ext autoreload
%autoreload 2

import os
from sqlalchemy import make_url
import pandas as pd

from utils_root_cause import *

from langchain_groq import ChatGroq
from typing import TypedDict, Literal, Optional
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
import json

import warnings
warnings.filterwarnings('ignore')


exp_llm = ChatGroq(
    groq_api_key=os.environ["GROQ_TOKEN"],
    model_name="openai/gpt-oss-20b", 
    temperature=0.78
)

DATABASE_URL_PUBLIC = os.getenv("DATABASE_URL_PUBLIC_RAG_DR")
DATABASE_URL = DATABASE_URL_PUBLIC.replace("postgres://", "postgresql://")
db_url = make_url(DATABASE_URL)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
auto_eval_output = get_auto_eval_output(db_url).iloc[:5]
print(auto_eval_output.shape)

auto_eval_output.head(n=2)

(5, 16)


,config_hash,dataset,embedding_model,top_n_retrieval,semantic_weight,answer_gen_llm,query,context,retrieved_content,same_context,retrieval_quality_score,rq_reasoning,referenced_answer,ai_answer,answer_quality_score,aq_reasoning
0,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,How to deposit a cheque issued to an associate...,Just have the associate sign the back and then...,Just have the associate sign the back and then...,true,3,The retrieved content reproduces the entire co...,Have the check reissued to the proper payee.Ju...,To deposit a cheque issued to an associate in ...,2,The AI’s answer correctly identifies that the ...
1,bcfe2f64d957b6c94a8cb20b52464cb9,FIQA Data,BAAI/bge-small-en-v1.5,1,0.5,llama-3.1-8b-instant,Can I send a money order from USPS as a business?,Sure you can. You can fill in whatever you wa...,Sure you can. You can fill in whatever you wa...,true,3,The retrieved content matches the context exac...,Sure you can. You can fill in whatever you wa...,"Yes, you can fill in your business name and ad...",2,The AI’s answer correctly states that a busine...


In [25]:
# =========================
# State Schema
# =========================

class RCAState(TypedDict):
    query: str
    referenced_content: str
    retrieved_content: str
    retrieval_quality_score: int
    rq_reasoning: str

    referenced_answer: str
    ai_answer: str
    answer_qualitys_score: int
    aq_reasoning: str

    new_retrieval_quality_score: int   
    new_answer_quality_score: int     
    root_cause_analysis: dict[str, list[str]]  # key is root cause, value is list of improvement suggestions

In [26]:
# =========================
# AI-as-Judge Reviewer
# =========================
async def review_evaluation_output(state: RCAState):
    rq_score_after_review = await review_sr_async(exp_llm, state['rq_reasoning'])
    aq_score_after_review = await review_sr_async(exp_llm, state['aq_reasoning'])

    return {
        "new_retrieval_quality_score": rq_score_after_review,
        "new_answer_quality_score": aq_score_after_review
    }

In [27]:
workflow = StateGraph(RCAState)

workflow.add_node("Evaluation Reviewer", review_evaluation_output)

workflow.add_edge(START, "Evaluation Reviewer")
workflow.add_edge("Evaluation Reviewer", END)

graph = workflow.compile()

In [34]:
async def run_rca_on_all(df, max_concurrent=2):
    sem = asyncio.Semaphore(max_concurrent)
    async def throttled_invoke(state):
        async with sem:
            return await graph.ainvoke(state)

    states = [
        {
            "query": row["query"],
            "referenced_content": row["context"],
            "retrieved_content": row["retrieved_content"],
            "retrieval_quality_score": int(row["retrieval_quality_score"]),
            "rq_reasoning": row["rq_reasoning"],
            "referenced_answer": row["referenced_answer"],
            "ai_answer": row["ai_answer"],
            "answer_qualitys_score": int(row["answer_quality_score"]),
            "aq_reasoning": row["aq_reasoning"],
            "root_cause_analysis": {}
        }
        for _, row in df.iterrows()
    ]
    return await asyncio.gather(*[throttled_invoke(s) for s in states])

rca_results = await run_rca_on_all(auto_eval_output)

In [ ]:
rca_results_df = pd.DataFrame(rca_results)
rca_results_df.head()

diff_df = rca_results_df[(rca_results_df["retrieval_quality_score"] != rca_results_df["new_retrieval_quality_score"]) \
                         | (rca_results_df["answer_qualitys_score"] != rca_results_df["new_answer_quality_score"])]
diff_df.head()

,query,referenced_content,retrieved_content,retrieval_quality_score,rq_reasoning,referenced_answer,ai_answer,answer_qualitys_score,aq_reasoning,new_retrieval_quality_score,new_answer_quality_score,root_cause_analysis
2,1 EIN doing business under multiple business n...,You're confusing a lot of things here. Company...,Sure you can. You can fill in whatever you wa...,1,The retrieved content discusses the need for a...,You're confusing a lot of things here. Company...,You can have one Employer Identification Numbe...,-1,The AI's answer directly addresses the user’s ...,2,-1,{}


In [46]:
print(diff_df["rq_reasoning"].values[0])

The retrieved content discusses the need for an EIN and a state‑issued DBA certificate to open a business account, which touches on the user’s question about using an EIN with multiple business names. However, it does not address the key context points such as merging entities, canceling old EINs, converting to an S‑Corp, or the fact that a single EIN can operate under multiple DBAs. Therefore it is relevant but lacks the critical contextual details.
